# Middleware

### middleware tightly controll what happened inside agent. middleware is useful for the following:
1. Tracking agent behavior with logging, analytics and debuging.
2. Transformaing prompts, tool selection and output formatting.
3. Adding retries, fallbacks, and early termination logic
4. Applying rate limits , guardrails and Pll detection.

In [ ]:
# executing normal model
import os
from langchain.chat_models import init_chat_model
from IPython.display import display,Markdown
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

# model=init_chat_model("groq:openai/gpt-oss-120b")

## Summarization Middleware


#### Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:
- long-running conversations that exceed context windows 
- Multi-turn dialogues with extensive history
- application where preserving full conversation context matters

In [5]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:openai/gpt-oss-120b",
            trigger=("messages", 10),
            keep=("messages", 4),
        ),
    ],
)

In [6]:
### run with thread id
config={"configurable": {"thread_id": "test-1"}}

In [11]:
# test data
questions=[

    "what is  2+2",
    "what is 10*5",
    "what is  100/4",
    "what is 15-7?",
    "what is 3*3",
    "what is 4*4",
    "what is 10/10",
    "what is 1/1",
]

for q in questions:
    response=agent.invoke({"messages": [HumanMessage(content=q)]}, config)
    print(f"messages: {response}")
    print(f"messages: {len(response['messages'])}")

messages: {'messages': [HumanMessage(content='what is  2+2', additional_kwargs={}, response_metadata={}, id='17e9df99-6965-4980-8dc5-79f8b7a005d5'), AIMessage(content='2\u202f+\u202f2\u202f=\u202f4.', additional_kwargs={'reasoning_content': 'User asks a simple math question: "what is 2+2". Provide answer: 4. Probably also can add brief explanation.'}, response_metadata={'token_usage': {'completion_tokens': 48, 'prompt_tokens': 78, 'total_tokens': 126, 'completion_time': 0.100731835, 'completion_tokens_details': {'reasoning_tokens': 29}, 'prompt_time': 0.004054, 'prompt_tokens_details': None, 'queue_time': 0.348430298, 'total_time': 0.104785835}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_dee443f41b', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a159b8-ca50-722f-a6c5-000835c748ab-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 78, 'output_tokens': 48, 'total_tokens': 126, 'o

## Token Size

based on the size of the token middleware will limit the model

In [16]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
@tool
def search_hotels(location :str)-> str:
    """ Search hotels - return response. """
    return f""" Hotels in {location}:
     1. Grand hotel - 5 star, $350/night, spa, pool, gym
     2. city Inn - 4 star, $180/night, bussiness center """

agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    tools=[search_hotels],
    checkpointer= InMemorySaver(), # this is for memory cache to preserve agent state
    middleware=[
        SummarizationMiddleware(
            model="groq:openai/gpt-oss-120b",
            trigger=("tokens", 2000),
            keep=("tokens", 200),
        ),
    ],
)

config={"configurable": {"thread_id": "test-1"}} #this is for a particular user

#count total tokens
def count_tokens(messages):
    total_chars=sum(len(str(m.content)) for m in messages)
    return total_chars//4

In [17]:
cities=["Pune", "Bengaluru", "HYD"]
for city in cities:
    response=agent.invoke({"messages" : [HumanMessage(content=f"best hotels in the city {city}")]},config)
    print(f"{(response["messages"])}")
    
    print("total tokens:",count_tokens(response['messages']))

[HumanMessage(content='best hotels in the city Pune', additional_kwargs={}, response_metadata={}, id='874b2381-512d-4b3c-bc48-583fd39390d7'), AIMessage(content='', additional_kwargs={'reasoning_content': 'User wants best hotels in Pune. Need to fetch data via search_hotels function. Provide location "Pune". Then format answer.', 'tool_calls': [{'id': 'fc_905d698c-8dc7-4c3e-ad83-186c10a3c742', 'function': {'arguments': '{"location":"Pune"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 126, 'total_tokens': 183, 'completion_time': 0.121400767, 'completion_tokens_details': {'reasoning_tokens': 28}, 'prompt_time': 0.0484564, 'prompt_tokens_details': None, 'queue_time': 0.220960327, 'total_time': 0.169857167}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_27194a498a', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a15a3e-c0